# Clima por estação para os picos de chamados (Alerta Rio + INMET via data.rio)

Este notebook baixa do **BigQuery público do data.rio** (projeto `datario`) as medições das estações de chuva e vento do Rio, agrega **por estação e por dia** e gera um arquivo `clima_estacoes_rio.zip` para enviar de volta.

**Como rodar (recomendado: Google Colab)**
1. Abra no Colab (Arquivo → Fazer upload de notebook).
2. Você precisa de um **projeto do Google Cloud** só para "pagar" as consultas. O **sandbox do BigQuery é gratuito** (1 TB de consulta/mês, sem cartão): acesse https://console.cloud.google.com/bigquery, crie um projeto e copie o **ID do projeto** (ex.: `meu-projeto-123456`).
3. Cole o ID em `PROJECT_ID` na célula de configuração e rode tudo (Ambiente de execução → Executar tudo).
4. No final, baixe `clima_estacoes_rio.zip` e mande na conversa.

O notebook **não altera nada** no data.rio: ele só lê. Antes de cada consulta ele faz um *dry run* e mostra quantos GB vai ler; se passar do limite definido, ele para e avisa.

**Rodando fora do Colab (Jupyter local):** instale `google-cloud-bigquery db-dtypes pandas` e rode no terminal `gcloud auth application-default login` antes.

In [ ]:
# ===== CONFIGURAÇÃO =====
PROJECT_ID = ""            # <- cole aqui o ID do seu projeto do Google Cloud
DATA_INI   = "2022-08-01"
DATA_FIM   = "2026-09-30"
LIMITE_GB_POR_CONSULTA = 60   # trava de segurança (o sandbox gratuito dá 1000 GB/mês)

# Datasets do data.rio a vasculhar (o notebook também tenta descobrir outros com 'clima' no nome)
DATASETS = ["clima_pluviometro", "clima_estacao_meteorologica"]

# Se a detecção automática errar alguma coluna, preencha aqui, por tabela. Ex.:
# OVERRIDES = {"clima_pluviometro.taxa_precipitacao_alertario": {"estacao": "id_estacao", "data": "data_particao"}}
OVERRIDES = {}
assert PROJECT_ID, "Preencha PROJECT_ID com o ID do seu projeto do Google Cloud"


In [ ]:
# ===== AUTENTICAÇÃO =====
import sys, subprocess
try:
    import google.colab  # noqa
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    from google.colab import auth
    auth.authenticate_user()
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-cloud-bigquery", "db-dtypes", "pandas"], check=False)

import pandas as pd, numpy as np, re, os, json, zipfile, datetime as dt
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)
print("Conectado ao BigQuery com o projeto", PROJECT_ID)


In [ ]:
# ===== DESCOBRIR TABELAS E COLUNAS =====
def q(sql, limite_gb=LIMITE_GB_POR_CONSULTA, verbose=True):
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    gb = client.query(sql, job_config=cfg).total_bytes_processed / 1e9
    if verbose: print(f"  dry run: {gb:,.2f} GB")
    if gb > limite_gb:
        raise RuntimeError(f"Consulta leria {gb:.1f} GB (> {limite_gb} GB). Aumente LIMITE_GB_POR_CONSULTA se quiser mesmo rodar.")
    return client.query(sql).to_dataframe()

datasets = list(DATASETS)
try:
    sch = q("SELECT schema_name FROM `datario.region-us.INFORMATION_SCHEMA.SCHEMATA`", verbose=False)
    extras = [s for s in sch.schema_name if 'clima' in s.lower() and s not in datasets]
    datasets += extras
    print("Datasets com 'clima' encontrados:", [s for s in sch.schema_name if 'clima' in s.lower()])
except Exception as e:
    print("Não consegui listar todos os datasets (seguindo com a lista padrão):", str(e)[:200])

cols = []
for ds in datasets:
    try:
        df = q(f"""SELECT table_name, column_name, data_type, is_partitioning_column, ordinal_position
                 FROM `datario.{ds}.INFORMATION_SCHEMA.COLUMNS`""", verbose=False)
        df.insert(0, "dataset", ds); cols.append(df)
    except Exception as e:
        print(f"Dataset {ds}: erro -> {str(e)[:200]}")
schema = pd.concat(cols, ignore_index=True).sort_values(["dataset","table_name","ordinal_position"])
os.makedirs("saida", exist_ok=True)
schema.to_csv("saida/schema_tabelas.csv", index=False)
for (ds, t), g in schema.groupby(["dataset","table_name"]):
    print(f"\n{ds}.{t}")
    print("   ", ", ".join(f"{r.column_name}:{r.data_type}{' [PART]' if r.is_partitioning_column=='YES' else ''}" for r in g.itertuples()))


In [ ]:
# ===== CLASSIFICAR TABELAS =====
TEMPO = ('data','hora','horario','time','particao','updated','extracted','timestamp')
def nomes(g): return dict(zip(g.column_name, g.data_type))

def col_estacao(c):
    for k in c:
        if k.lower() in ('id_estacao','estacao_id','codigo_estacao','id'): return k
    for k in c:
        if 'estacao' in k.lower() and 'lat' not in k.lower() and 'lon' not in k.lower(): return k
    return None

def col_data(c, part):
    # 1) DATE "de medição"; 2) DATETIME/TIMESTAMP de medição; 3) partição
    pref = ['data','data_medicao','data_hora','datahora','data_particao']
    for p in pref:
        if p in c and c[p] in ('DATE','DATETIME','TIMESTAMP'): return p
    for k,t in c.items():
        if t in ('DATETIME','TIMESTAMP') and not any(x in k.lower() for x in ('update','extract','ingest','carga')): return k
    return part

def expr_data(nome, tipo):
    if tipo == 'TIMESTAMP': return f"DATE({nome}, 'America/Sao_Paulo')"
    if tipo in ('DATETIME','DATE'): return f"DATE({nome})"
    return f"SAFE_CAST({nome} AS DATE)"

def classifica(ds, t, g):
    c = nomes(g); low = {k.lower():k for k in c}
    part = next((r.column_name for r in g.itertuples() if r.is_partitioning_column=='YES'), None)
    tem_latlon = any('lat' in k for k in low) and any('lon' in k for k in low)
    chuva15 = [low[k] for k in low if re.search(r'chuva.*15', k) or re.search(r'15.*min', k) and 'chuva' in k]
    chuva1h = [low[k] for k in low if re.search(r'chuva.*_1_?h', k)]
    precip  = [low[k] for k in low if ('precipita' in k or k in ('chuva','acumulado_chuva')) and not re.search(r'_(4|24|96)_?h|mes', k)]
    rajada  = [low[k] for k in low if 'rajada' in k]
    vento   = [low[k] for k in low if 'vento' in k and ('veloc' in k or 'velocidade' in k) and 'rajada' not in k]
    est = col_estacao(c); dcol = col_data(c, part)
    ov = OVERRIDES.get(f"{ds}.{t}", {})
    est = ov.get('estacao', est); dcol = ov.get('data', dcol)
    if (chuva15 or chuva1h or precip or rajada or vento) and est and dcol:
        tipo = 'medicao'
    elif tem_latlon and est:
        tipo = 'estacoes'
    else:
        tipo = 'ignorar'
    return dict(dataset=ds, tabela=t, tipo=tipo, estacao=est, data=dcol, tipo_data=c.get(dcol), particao=part,
                chuva15=chuva15, chuva1h=chuva1h, precip=[p for p in precip if p not in chuva15+chuva1h],
                rajada=rajada, vento=vento)

plano = [classifica(ds, t, g) for (ds, t), g in schema.groupby(["dataset","table_name"])]
plano_df = pd.DataFrame(plano)
pd.set_option('display.max_colwidth', 80); pd.set_option('display.width', 200)
display(plano_df) if 'display' in dir() else print(plano_df.to_string())


In [ ]:
# ===== BAIXAR ESTAÇÕES (tabelas pequenas, inteiras) =====
for p in plano:
    if p['tipo'] != 'estacoes': continue
    fq = f"`datario.{p['dataset']}.{p['tabela']}`"
    print("Estações:", fq)
    try:
        df = q(f"SELECT * FROM {fq}")
        nome = f"saida/estacoes__{p['dataset']}__{p['tabela']}.csv"
        df.to_csv(nome, index=False); print(f"  {len(df)} linhas -> {nome}")
    except Exception as e:
        print("  erro:", str(e)[:300])


In [ ]:
# ===== AGREGAR MEDIÇÕES POR ESTAÇÃO × DIA =====
F = lambda c: f"SAFE_CAST({c} AS FLOAT64)"
resumo = []
for p in plano:
    if p['tipo'] != 'medicao': continue
    fq = f"`datario.{p['dataset']}.{p['tabela']}`"
    d = expr_data(p['data'], p['tipo_data'])
    aggs = []
    for c in p['chuva15']: aggs += [f"SUM({F(c)}) AS {c}__soma_dia", f"MAX({F(c)}) AS {c}__max"]
    for c in p['chuva1h']: aggs += [f"MAX({F(c)}) AS {c}__max"]
    for c in p['precip']:  aggs += [f"SUM({F(c)}) AS {c}__soma_dia", f"MAX({F(c)}) AS {c}__max"]
    for c in p['rajada']:  aggs += [f"MAX({F(c)}) AS {c}__max"]
    for c in p['vento']:   aggs += [f"MAX({F(c)}) AS {c}__max", f"AVG({F(c)}) AS {c}__media"]
    filtro = f"{d} BETWEEN '{DATA_INI}' AND '{DATA_FIM}'"
    if p['particao'] and p['particao'] != p['data']:
        filtro = f"{p['particao']} BETWEEN DATE_SUB('{DATA_INI}', INTERVAL 1 DAY) AND DATE_ADD('{DATA_FIM}', INTERVAL 1 DAY) AND " + filtro
    sql = f"""SELECT CAST({p['estacao']} AS STRING) AS id_estacao, {d} AS data, COUNT(*) AS n_registros,
               {', '.join(aggs)}
             FROM {fq} WHERE {filtro}
             GROUP BY 1, 2 ORDER BY 1, 2"""
    print(f"\nMedições: {fq}  (data = {p['data']}, estação = {p['estacao']})")
    try:
        df = q(sql)
        nome = f"saida/diario__{p['dataset']}__{p['tabela']}.csv"
        df.to_csv(nome, index=False)
        print(f"  {len(df):,} estação-dias, {df.id_estacao.nunique()} estações, {df.data.min()} a {df.data.max()} -> {nome}")
        resumo.append(dict(tabela=f"{p['dataset']}.{p['tabela']}", linhas=len(df), estacoes=df.id_estacao.nunique(),
                           ini=str(df.data.min()), fim=str(df.data.max()), n_registros_mediano=float(df.n_registros.median()),
                           colunas=list(df.columns), sql=sql))
    except Exception as e:
        print("  erro:", str(e)[:400])
        resumo.append(dict(tabela=f"{p['dataset']}.{p['tabela']}", erro=str(e)[:400], sql=sql))


In [ ]:
# ===== AMOSTRA BRUTA (para eu conferir unidades e duplicatas) =====
# 200 linhas cruas de cada tabela de medição, num dia conhecido de temporal (29/07/2026) e num dia de chuva forte (05/04/2025)
for p in plano:
    if p['tipo'] != 'medicao': continue
    fq = f"`datario.{p['dataset']}.{p['tabela']}`"; d = expr_data(p['data'], p['tipo_data'])
    filtro = f"{d} IN ('2026-07-29','2025-04-05')"
    if p['particao'] and p['particao'] != p['data']:
        filtro = f"{p['particao']} IN ('2026-07-28','2026-07-29','2026-07-30','2025-04-04','2025-04-05','2025-04-06') AND " + filtro
    try:
        df = q(f"SELECT * FROM {fq} WHERE {filtro} LIMIT 200")
        df.to_csv(f"saida/amostra__{p['dataset']}__{p['tabela']}.csv", index=False)
        print(f"{fq}: {len(df)} linhas de amostra")
    except Exception as e:
        print(f"{fq}: erro na amostra -> {str(e)[:200]}")


In [ ]:
# ===== CHECAGEM RÁPIDA =====
for r in resumo:
    if 'erro' in r: print("ERRO", r['tabela'], r['erro'][:150]); continue
    df = pd.read_csv(f"saida/diario__{r['tabela'].replace('.', '__')}.csv", parse_dates=['data'])
    print(f"\n{r['tabela']}: {r['estacoes']} estações, {r['ini']} a {r['fim']}, registros/dia mediano = {r['n_registros_mediano']:.0f}")
    num = [c for c in df.columns if '__' in c]
    dias = df[df.data.isin(pd.to_datetime(['2025-04-05','2026-07-29']))]
    if len(dias): print(dias.groupby('data')[num].max().round(1).to_string())

with open("saida/resumo.json", "w") as f:
    json.dump(dict(gerado_em=str(dt.datetime.now()), data_ini=DATA_INI, data_fim=DATA_FIM, plano=plano, resumo=resumo), f, ensure_ascii=False, indent=1, default=str)


In [ ]:
# ===== EMPACOTAR E BAIXAR =====
with zipfile.ZipFile("clima_estacoes_rio.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir("saida")):
        z.write(os.path.join("saida", fn), fn)
print("Arquivos no zip:"); [print("  ", fn, f"{os.path.getsize(os.path.join('saida', fn))/1e6:.2f} MB") for fn in sorted(os.listdir("saida"))]
print(f"\nclima_estacoes_rio.zip: {os.path.getsize('clima_estacoes_rio.zip')/1e6:.1f} MB")
if EM_COLAB:
    from google.colab import files
    files.download("clima_estacoes_rio.zip")


### Se algo der errado
- **"Access Denied" / "billing"**: confira o `PROJECT_ID` e se a API do BigQuery está ativada no projeto (o console oferece ativar na primeira visita).
- **Tabela classificada errado** na célula *Classificar tabelas*: preencha `OVERRIDES` na configuração com o nome certo da coluna de estação ou de data e rode de novo a partir dali.
- **Trava de GB acionada**: aumente `LIMITE_GB_POR_CONSULTA`. Consultas com filtro de partição costumam ler poucos GB.
- Em qualquer caso, mande o `schema_tabelas.csv` (sai mesmo que o resto falhe) e a mensagem de erro; eu ajusto o notebook.